## Importing modules

In [17]:
from pypdf import PdfReader
import string
import re
from bs4 import BeautifulSoup as bs
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import sentence_transformers
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

## Reading the Document

In [18]:
df=PdfReader('power bi.pdf')
text=''
for i in df.pages:
    text += i.extract_text()
print(text)

Introducing
Microsoft 
Power BI
Alberto Ferrari and Marco RussoPUBLISHED BY 
Microsoft Press 
A division of Microsoft Corporation 
One Microsoft Way 
Redmond, Washington 98052-6399 
Copyright © 2016 by Microsoft Corporation 
All rights reserved. No part of the contents of this book may be reproduced or transmitted in any 
form or by any means without the written permission of the publisher. 
ISBN: 978-1-5093-0228-4 
Microsoft Press books are available through booksellers and distributors worldwide. If you need 
support related to this book, email Microsoft Press Support at mspinput@microsoft.com. Please tell us 
what you think of this book at http://aka.ms/tellpress. 
This book is provided “as-is” and expresses the author’s views and opinions. The views, opinions and 
information expressed in this book, including URL and other Internet website references, may change 
without notice. 
Some examples depicted herein are provided for illustration only and are fictitious. No real associatio

## NLP Preprocess Techiques

In [19]:
text=text.lower()
text=bs(text,"html.parser").get_text()
text=text.translate(str.maketrans('','',string.punctuation))
text=re.sub(r"\d+",'',text)
text=re.sub(r"\s+",' ',text).strip()

tokens=word_tokenize(text)
print(len(tokens))
sentence=sent_tokenize(text)
print(len(sentence))
stop_words=set(stopwords.words('english'))
tokens=[word for word in tokens if word not in stop_words]
lemmatizer=WordNetLemmatizer()
tokens=[lemmatizer.lemmatize(word) for word in tokens]
text=" ".join(tokens)

51985
1


## Converting the text into the chunks

In [20]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunk=splitter.split_text(text)
print(len(chunk))

451


In [21]:
for i,chunks in enumerate(chunk):
    print("=" * 50)
    print("chunks",i+1)
    print("="*50)
    print(chunks)

chunks 1
introducing microsoft power bi alberto ferrari marco russopublished microsoft press division microsoft corporation one microsoft way redmond washington copyright © microsoft corporation right reserved part content book may reproduced transmitted form mean without written permission publisher isbn microsoft press book available bookseller distributor worldwide need support related book email microsoft press support mspinputmicrosoftcom please tell u think book httpakamstellpress book provided “
chunks 2
u think book httpakamstellpress book provided “ asis ” express author ’ view opinion view opinion information expressed book including url internet website reference may change without notice example depicted herein provided illustration fictitious real association connection intended inferred microsoft trademark listed httpwwwmicrosoftcom “ trademark ” webpage trademark microsoft group company mark property respective owner acquisition developmental editor rosemary caperton edi

## Using embedding method

In [22]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## FAISS  to store the data

In [23]:
vector_db=FAISS.from_texts(
    texts=chunk,
    embedding=embedding_model
)

In [24]:
vector_db.save_local('power_index')

In [25]:
vector_db=FAISS.load_local(
    "power_index",
    embedding_model,
    allow_dangerous_deserialization=True

)

## Asking question to the model

In [26]:
ques='define powerbi'
search=vector_db.similarity_search(ques,k=4)

In [27]:
search

[Document(id='f42bd173-8774-4127-a582-2afef8ec366a', metadata={}, page_content='power bi power pivot excel expert trainer alberto ferrari marco russo ’ easy • sign receive special offer microsoft press • watch inbox exclusive email offer first week september visit microsoftpressstorecompowerbi get started even better ’ receive code save next purchase microsoft press store learn power bi powerbimicrosoftcom discount code valid single book ebook purchase microsoftpressstorecom code combined ebook deal day official microsoft practice test fulfilled measureup offer offer'),
 Document(id='09ed4463-e732-4243-a0cf-5e2c90131199', metadata={}, page_content='introducing microsoft power bi alberto ferrari marco russopublished microsoft press division microsoft corporation one microsoft way redmond washington copyright © microsoft corporation right reserved part content book may reproduced transmitted form mean without written permission publisher isbn microsoft press book available bookseller dis

In [28]:
for doc in search:
    print("="*50)
    print("search answer")
    print("="*50)
    print(doc.page_content)

search answer
power bi power pivot excel expert trainer alberto ferrari marco russo ’ easy • sign receive special offer microsoft press • watch inbox exclusive email offer first week september visit microsoftpressstorecompowerbi get started even better ’ receive code save next purchase microsoft press store learn power bi powerbimicrosoftcom discount code valid single book ebook purchase microsoftpressstorecom code combined ebook deal day official microsoft practice test fulfilled measureup offer offer
search answer
introducing microsoft power bi alberto ferrari marco russopublished microsoft press division microsoft corporation one microsoft way redmond washington copyright © microsoft corporation right reserved part content book may reproduced transmitted form mean without written permission publisher isbn microsoft press book available bookseller distributor worldwide need support related book email microsoft press support mspinputmicrosoftcom please tell u think book httpakamstellp

## LLM model

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=" Groq API Key "
)

In [30]:
context="\n".join([doc.page_content for doc in search])

In [31]:
prompt = f"""
Answer the question only from the given context.

Context:
{context}

Question:
{ques}
"""

## Response form the model and the final output

In [32]:
response=llm.invoke(prompt)
print("Response: ")
print(response.content)

Response: 
The context does not provide a direct definition of Power BI. However, based on the given information, Power BI appears to be a business analytics service by Microsoft that allows users to upload workbooks, connect to data sources, and generate reports and visualizations. It seems to be related to data analysis and visualization, and is available for download from the Power BI website.
